In [1]:
# ######### used this part for fixing problems running on ARC #

import os


os.environ['HF_HOME'] = '/projects/abbott_lab/Users/ishtiaque/hfmodels/'
os.environ['HF_HUB_CACHE'] = '/projects/abbott_lab/Users/ishtiaque/hfmodels/'
os.environ['XDG_CACHE_HOME'] = '/projects/abbott_lab/Users/ishtiaque/hfmodels/'
os.environ['NB_USER'] = 'ishtiahmed'#'ishtiaqueahmedk'
os.environ['TRANSFORMERS_CACHE'] = '/projects/abbott_lab/Users/ishtiaque/hfmodels/'
os.environ['HF_DATASETS_CACHE'] = '/projects/abbott_lab/Users/ishtiaque/hfmodels/'




In [2]:
import os
import json
from PIL import Image, ImageDraw, ImageFont
from pathlib import Path
from typing import Dict, List, Tuple, Optional
import random

import torch
import os 
import json
import random
from tqdm import tqdm
from collections import Counter 

import numpy as np
import torch
import torchvision.transforms as T


In [3]:
import sys
sys.path.append('/projects/abbott_lab/Users/ishtiaque/hfmodels/DeepSeek-VL2')

import torch
from transformers import AutoModelForCausalLM

from deepseek_vl2.models import DeepseekVLV2Processor, DeepseekVLV2ForCausalLM
from deepseek_vl2.utils.io import load_pil_images


/projects/abbott_lab/Users/ishtiaque/env/deepseek_vl_2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/projects/abbott_lab/Users/ishtiaque/env/deepseek_vl_2/lib/python3.10/site-packages/transformers/utils/hub.py:124: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


Python version is above 3.10, patching the collections module.


In [4]:


# specify the path to the model
model_path = "deepseek-ai/deepseek-vl2-tiny"
# model_path = "deepseek-ai/deepseek-vl2-small"
vl_chat_processor: DeepseekVLV2Processor = DeepseekVLV2Processor.from_pretrained(model_path)
tokenizer = vl_chat_processor.tokenizer

vl_gpt: DeepseekVLV2ForCausalLM = AutoModelForCausalLM.from_pretrained(model_path, trust_remote_code=True)
vl_gpt = vl_gpt.to(torch.bfloat16).cuda().eval()



/projects/abbott_lab/Users/ishtiaque/env/deepseek_vl_2/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Add pad token = ['<｜▁pad▁｜>'] to the tokenizer
<｜▁pad▁｜>:2
Add image token = ['<image>'] to the tokenizer
<image>:128815
Add grounding-related tokens = ['<|ref|>', '<|/ref|>', '<|det|>', '<|/det|>', '<|grounding|>'] to the tokenizer with input_ids
<|ref|>:128816
<|/ref|>:128817
<|det|>:128818
<|/det|>:128819
<|grounding|>:128820
Add chat tokens = ['<|User|>', '<|Assistant|>'] to the tokenizer with input_ids
<|User|>:128821
<|Assistant|>:128822



In [5]:
# # multiple images (or in-context learning) conversation example
# local_img_paths = ["baddooor.jpg", "bagghhuu.jpg", "bill.jpg", "sing.jpg", "zxcasd.jpg"]
# images = []
# for i, local_img_path in enumerate (local_img_paths):
#     local_img_path = f"/home/ishtiaqueahmedk/Research/VLM/fine_grained/multi_image_prac/1/{local_img_path}"
#     images.append(local_img_path)


#             # "/projects/abbott_lab/Users/ishtiaque/hfmodels/DeepSeek-VL2/images/visual_grounding_1.jpeg",
#             # "/projects/abbott_lab/Users/ishtiaque/hfmodels/DeepSeek-VL2/images/visual_grounding_2.jpg",
#             # "/projects/abbott_lab/Users/ishtiaque/hfmodels/DeepSeek-VL2/images/visual_grounding_3.png",


# # multiple images/interleaved image-text
# query = "Mention which image shows a monkey."
# conversation = [
#     {
#         "role": "<|User|>",
#         "content": "This is image_1: <image>\n"
#                    "This is image_2: <image>\n"
#                    "This is image_3: <image>\n" 
#                    "This is image_4: <image>\n"
#                    f"This is image_5: <image>\n {query}",
#         "images": [
#             images[0],
#             images[1],
#             images[2],
#             images[3],
#             images[4]
#         ],
#     },
#     {"role": "<|Assistant|>", "content": ""}
# ]

# # load images and prepare for inputs
# pil_images = load_pil_images(conversation)
# prepare_inputs = vl_chat_processor(
#     conversations=conversation,
#     images=pil_images,
#     force_batchify=True,
#     system_prompt=""
# ).to(vl_gpt.device)

# # run image encoder to get the image embeddings
# inputs_embeds = vl_gpt.prepare_inputs_embeds(**prepare_inputs)

# # run the model to get the response
# outputs = vl_gpt.language.generate(
#     inputs_embeds=inputs_embeds,
#     attention_mask=prepare_inputs.attention_mask,
#     pad_token_id=tokenizer.eos_token_id,
#     bos_token_id=tokenizer.bos_token_id,
#     eos_token_id=tokenizer.eos_token_id,
#     max_new_tokens=2048,
#     do_sample=False,
#     use_cache=True
# )

# answer = tokenizer.decode(outputs[0].cpu().tolist(), skip_special_tokens=False)
# print(f"{prepare_inputs['sft_format'][0]}", answer)
# answer2 = answer.replace("<｜end▁of▁sentence｜>", "")
# answer2



In [6]:
def get_answer(answ_img_paths, query):


    
    # multiple images/interleaved image-text
    conversation = [
        {
            "role": "<|User|>",
            "content": "This is image_1: <image>\n"
                       "This is image_2: <image>\n"
                       "This is image_3: <image>\n" 
                       f"This is image_4: <image>\n {query}",
            "images": [
                answ_img_paths[0],
                answ_img_paths[1],
                answ_img_paths[2],
                answ_img_paths[3]
            ],
        },
        {"role": "<|Assistant|>", "content": ""}
    ]
    
    # load images and prepare for inputs
    pil_images = load_pil_images(conversation)
    prepare_inputs = vl_chat_processor(
        conversations=conversation,
        images=pil_images,
        force_batchify=True,
        system_prompt=""
    ).to(vl_gpt.device)
    
    # run image encoder to get the image embeddings
    inputs_embeds = vl_gpt.prepare_inputs_embeds(**prepare_inputs)
    
    # run the model to get the response
    outputs = vl_gpt.language.generate(
        inputs_embeds=inputs_embeds,
        attention_mask=prepare_inputs.attention_mask,
        pad_token_id=tokenizer.eos_token_id,
        bos_token_id=tokenizer.bos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        max_new_tokens=2048,
        do_sample=False,
        use_cache=True
    )
    
    answer = tokenizer.decode(outputs[0].cpu().tolist(), skip_special_tokens=False)
    # print(f"{prepare_inputs['sft_format'][0]}", answer)
    answer2 = answer.replace("<｜end▁of▁sentence｜>", "")
    return answer2
    


In [7]:
def get_bird_images(images_folder):
# Path to the folder containing bird subfolders
    # images_folder = '/projects/abbott_lab/Users/ishtiaque/datasets/CUB_200_2011/images'
    
    # Dictionary to store bird names and their corresponding image paths
    bird_images = {}
    
    # Iterate over each subfolder in the images folder
    for folder in os.listdir(images_folder):
        bird_name = folder.split(".")[-1]  # Extract bird name from folder name
        folder_path = os.path.join(images_folder, folder)  # Path to the bird's folder
        
        # Initialize an empty list to store image paths for the current bird
        image_paths = []
        
        # Iterate over the image files in the bird's folder
        for image_file in os.listdir(folder_path):
            image_path = os.path.join(folder_path, image_file)  # Full path to the image file
            image_paths.append(image_path)  # Store the image path
        
        # Store the list of image paths in the dictionary under the bird's name
        bird_images[bird_name] = image_paths
    
    # Now bird_images contains a dictionary where the keys are bird names and the values are lists of image paths
    print(len(bird_images))
    return bird_images



In [8]:

def get_json_data(json_file_name):

    # negated_questions, modified_new_cub_class_descriptions_full_fake, #modified_new_cub_class_descriptionsx #modified_mcqs_description_only2, modified_mcqs_description_only

    # For Task 1 type 1: 
    with open(json_file_name, "r") as file: 
        json_data = json.load(file)
    print(len(json_data))
    return json_data


In [9]:
def get_map_dict(json_data):
    
    #create dictionary ishti

    captions_mcqid_pair_dict = {}
    class_freq_dict = {}
    
    for mcq in json_data:
        
        options = mcq['options'] # get the answer choices dictionary of the first mcq
        correct_option_description = {options[mcq['correct_answer']]} # get the correct answer caption
        class_name = mcq['mcq_id']
    
        # make sure that there is only one correct caption
        assert len(correct_option_description)==1, "More than one correct answer!!"
    
        #convert to string
        correct_option_description = next(iter(correct_option_description))
    
        if correct_option_description in captions_mcqid_pair_dict: # if caption already in dict
            
            # get the existing mcq_id and make sure it matches
            exist_class_name = captions_mcqid_pair_dict[correct_option_description] 
            
            # make sure mcq_id matches
            assert exist_class_name == class_name, f"mismatch in class name: [{exist_class_name}] and [{class_name}]"
            class_freq_dict [class_name] = class_freq_dict [class_name] + 1
            
    
                
        else:
            captions_mcqid_pair_dict[correct_option_description] = class_name
    
            class_freq_dict [class_name] = 1 

    return captions_mcqid_pair_dict
    
    
    # print(f"Successfully created class_names dictionary with {len(captions_mcqid_pair_dict)} Class entries from JSON file:\n {filename}.\n")
    
    # print(captions_mcqid_pair_dict)
    # print(class_freq_dict)

    

In [10]:
def get_medium_hard_data(json_data):
    
# Use this if the json file contain the medium and hard categories
    medium_data = []
    hard_data = []

    data_counter = 0
    use_partial = False#True#False
    if use_partial:
        print("using limited data for debugging")
    
    for data in json_data:
        if data['difficulty'] == "Medium":
            medium_data.append(data)
        else:
            hard_data.append(data)

        

        data_counter = data_counter + 1
        if (data_counter>50) and use_partial:
            print(f"stopping at data = {data_counter}")
            break
        
    print(len(medium_data))
    print(len(hard_data))
    return medium_data, hard_data
    

In [11]:
def run_eval(data_partition):
    

    # Counters for distribution
    true_distribution = Counter()
    predicted_distribution = Counter()
    
    results = []
    
    for i, item in tqdm(enumerate(data_partition)): # for easy part json_data, for medium_data, for hard_data 
        mcq_id = item['mcq_id']
        question = item['question']
        options = item['options']
        correct_answer = item['correct_answer']
    
        if mcq_id not in bird_images or not bird_images[mcq_id]:
            print(f"No image for {mcq_id}") 
            continue
    
        image_paths = bird_images[mcq_id][:5]
    
        # Format the prompt
        description = options[correct_answer]
        formatted_prompt = f"Which image (A, B, C or D) matches best with this description: {description}?\n"
    
    
        
    
        # Final prompt
        prompt = f""" Your answer or response must ONLY be a single index ('A', 'B', 'C', 'D'). Do not response with any other text. 
    
        {formatted_prompt}
    
        Answer: ('A', 'B', 'C', 'D')"""
    
        # Run the model
        for image_path in image_paths:
    
            #get the image paths for the four answer options
            answ_img_paths = []
            for k in ['A', 'B', 'C', 'D']:  # ['D', 'C', 'B', 'A'] for position bias checking ['A', 'B', 'C', 'D']
                # formatted_prompt += f"{k}. {options[k]}\n"
                
                if (correct_answer == k):
                    answ_img_paths.append(image_path)
                    continue
                description = options[k]
                answ_mcq_id = captions_mcqid_pair_dict[description]
                answ_mcq_img_path = bird_images[answ_mcq_id][0]
                answ_img_paths.append(answ_mcq_img_path)
                
            # print(answ_img_paths)
            
    
                
            model_output = get_answer(answ_img_paths, prompt)
            # print("Model Output: ", model_output)
    
            # Extract predicted answer (basic string search, can refine)
            predicted_answer = None
            for option in ['A', 'B', 'C', 'D']:
                if f"{option}" in model_output or f"{option}." in model_output:
                    predicted_answer = option
    
            # Update counters
            true_distribution[correct_answer] += 1
            if predicted_answer:
                predicted_distribution[predicted_answer] += 1
            
            results.append({
                'mcq_id': mcq_id,
                'image_path': image_path,
                'prompt': prompt,
                'model_output': model_output,
                'predicted_answer': predicted_answer,
                'correct_answer': correct_answer,
                'is_correct': predicted_answer == correct_answer
            })
    
    # Accuracy summary 
    print(f"Results for file: {json_file_name}")
    
    correct = sum(r['is_correct'] for r in results if r['predicted_answer'] is not None)
    total = len(results)
    print(f"Accuracy: {correct}/{total} = {correct / total:.2%}") 
    
    # Print distributions
    print("True Option Distribution:", dict(true_distribution))
    print("Predicted Option Distribution:", dict(predicted_distribution))

    return prompt, answ_img_paths

Image version--
Results for file: new_cub_with_class_descriptions.json
Accuracy: 601/1000 = 60.10%
True Option Distribution: {'C': 285, 'A': 270, 'B': 220, 'D': 225}
Predicted Option Distribution: {'C': 272, 'B': 251, 'A': 147, 'D': 330}



In [12]:

# json_files_list = ['negated_questions', 'new_cub_class_descriptions','new_cub_with_class_descriptions', 'modified_new_cub_class_descriptions_full_fake', 'modified_new_cub_class_descriptions_partial_fake', 'incorrect_class_correct_description', 'correct_class_incorrect_description']

bird_folder = "/projects/abbott_lab/Users/ishtiaque/datasets/CUB_200_2011/images"
food_folder = "/projects/abbott_lab/Users/ishtiaque/datasets/food-101/food-101/images"
aircraft_folder = "/projects/abbott_lab/Users/ishtiaque/datasets/fgvc-aircraft-2013b/data/test"
dogs_folder = "/projects/abbott_lab/Users/ishtiaque/datasets/stanford-dogs/images/Images"
car_folder = "/projects/abbott_lab/Users/ishtiaque/datasets/stanford-cars/train"

bird_list = ["new_cub_class_descriptions", "new_cub_class_descriptions_task_0a_with_class_baseline"]
food_list = ["new_food_class_descriptions", "new_food_class_descriptions_task_0a_with_class_baseline"]
aircraft_list = ["new_aircraft_class_descriptions", "new_aircraft_class_descriptions_task_0a_with_class_baseline"]
dogs_list = ["new_dogs_class_descriptions", "new_dogs_class_descriptions_task_0a_with_class_baseline"]
car_list = ["new_car_class_descriptions", "new_car_class_descriptions_task_0a_with_class_baseline"]


# json_files_list = food_list  #################### change!!!!!!!!!!!!!!!!!!!!
all_lists = [bird_list, food_list, aircraft_list, dogs_list, car_list]
folder_lists = [bird_folder, food_folder, aircraft_folder, dogs_folder, car_folder]

for json_files_list, images_folder in zip(all_lists, folder_lists):

    print(f"Number of JSON files: {len(json_files_list)}\n")

    bird_images = get_bird_images(images_folder)
    
    for json_file_name in json_files_list:
        
        json_file_name = f"/home/ishtiaqueahmedk/Research/VLM/fine_grained/mcq_json files/{json_file_name}.json"
        json_data = get_json_data(json_file_name)
    
        # json_data[0]
        # json_data[1]
    
        medium_data, hard_data = get_medium_hard_data(json_data)

        captions_mcqid_pair_dict = get_map_dict(json_data)
        
        print("\n----Medium----")
        prompt, answ_img_paths = run_eval(medium_data)
        print("\n----Hard----")
        prompt, answ_img_paths = run_eval(hard_data)
    
    
    


Number of JSON files: 2

200
400
200
200

----Medium----


0it [00:00, ?it/s]You're using a LlamaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
200it [04:10,  1.25s/it]


Results for file: /home/ishtiaqueahmedk/Research/VLM/fine_grained/mcq_json files/new_cub_class_descriptions.json
Accuracy: 382/1000 = 38.20%
True Option Distribution: {'D': 245, 'C': 220, 'B': 235, 'A': 300}
Predicted Option Distribution: {'A': 896, 'D': 69, 'B': 34, 'C': 1}

----Hard----


200it [04:02,  1.21s/it]


Results for file: /home/ishtiaqueahmedk/Research/VLM/fine_grained/mcq_json files/new_cub_class_descriptions.json
Accuracy: 275/1000 = 27.50%
True Option Distribution: {'D': 275, 'A': 260, 'B': 215, 'C': 250}
Predicted Option Distribution: {'A': 952, 'D': 28, 'B': 20}
400
200
200

----Medium----


...

Results for file: /home/ishtiaqueahmedk/Research/VLM/fine_grained/mcq_json files/new_car_class_descriptions.json
Accuracy: 247/980 = 25.20%
True Option Distribution: {'A': 245, 'D': 250, 'B': 245, 'C': 240}
Predicted Option Distribution: {'A': 947, 'D': 24, 'B': 6, 'C': 3}
392
196
196

----Medium----


196it [07:41,  2.35s/it]


Results for file: /home/ishtiaqueahmedk/Research/VLM/fine_grained/mcq_json files/new_car_class_descriptions_task_0a_with_class_baseline.json
Accuracy: 258/980 = 26.33%
True Option Distribution: {'B': 250, 'C': 300, 'D': 250, 'A': 180}
Predicted Option Distribution: {'D': 881, 'A': 99}

----Hard----


196it [07:37,  2.34s/it]

Results for file: /home/ishtiaqueahmedk/Research/VLM/fine_grained/mcq_json files/new_car_class_descriptions_task_0a_with_class_baseline.json
Accuracy: 269/980 = 27.45%
True Option Distribution: {'A': 245, 'D': 250, 'B': 245, 'C': 240}
Predicted Option Distribution: {'D': 817, 'A': 163}
